In [8]:
import os
import sys
import difflib
import importlib

PARSER_MODULE_NAME = 'pds_parser'
print("--- Initializing Test Environment ---")
current_working_directory = os.getcwd()
if current_working_directory not in sys.path:
    sys.path.insert(0, current_working_directory)
print(f"Current working directory: {current_working_directory}")

if PARSER_MODULE_NAME in sys.modules:
    try:
        parser_module_reloaded = importlib.reload(sys.modules[PARSER_MODULE_NAME])
        print(f"--- Module '{PARSER_MODULE_NAME}' reloaded. ---")
        from pds_parser import PdsParser, PdsBlock, PdsKeyValuePair, PdsComment, PdsBlankLine, PdsList
    except Exception as e:
        print(f"--- FAILED to reload module '{PARSER_MODULE_NAME}': {e} ---")
        from pds_parser import PdsParser, PdsBlock, PdsKeyValuePair, PdsComment, PdsBlankLine, PdsList
else:
    from pds_parser import PdsParser, PdsBlock, PdsKeyValuePair, PdsComment, PdsBlankLine, PdsList
    print(f"--- Module '{PARSER_MODULE_NAME}' imported fresh. ---")

print(f"--- Using PdsParser from: {PdsParser.__module__}.py ---")
print("-" * 80)

test_content = """
################
# WITCH EVENTS #
################

namespace = witch

#witch.1001-1999 - Guardian coverts ward
#witch.2001-2899 - Convert to witchcraft scheme

witch.1001 = { #by Mathilda Bjarnehed
    hidden = yes
    
    trigger = {
        is_witch_trigger = no
        any_relation = {
            type = guardian
            is_witch_trigger = yes
        }
    }

    immediate = {
        save_scope_as = child
        if = { # Conditional block
            limit = {
                is_ai = yes
                exists = house
                house = {
                    has_house_modifier = witch_coven
                    house_head = { is_ai = yes }
                }
                any_relation = {
                    type = guardian
                    is_ai = yes
                }
            }
            child_witch_conversion_success_effect = yes
        }
        else = {
            random_relation = { type = guardian trigger_event = witch.1002 }
        }
    }
}   

scripted_trigger witch_1002_allow_reveal_outcome_trigger = {
    exists = scope:child.liege
    scope:guardian = {
        NOT = { this = scope:child.liege }
        any_secret = {
            secret_type = secret_witch
            OR = {
                NOT = { is_known_by = scope:child }
                NOT = { is_known_by = scope:child.liege }
            }
        }
    }
}


# Standard Values
@pos_compat_high = 30
@pos_compat_medium = 15
@pos_compat_low = 5

# INTRIGUE OUTCOMES
education_intrigue_1 = {
    minimum_age = 16
    intrigue = 2
    category = education
    monthly_intrigue_lifestyle_xp_gain_mult = 0.1
    
    ruler_designer_cost = 0
    
    culture_modifier = {
        parameter = poorly_educated_leaders_distrusted
        feudal_government_opinion = -10
    }
    
    desc = {
        first_valid = {
            triggered_desc = {
                trigger = {
                    NOT = { exists = this }
                }
                desc = trait_education_intrigue_1_desc
            }
            desc = trait_education_intrigue_1_character_desc
        }
    }

    group = education_intrigue
    level = 1
}
education_intrigue_2 = {
    minimum_age = 16
    intrigue = 4
    category = education
    monthly_intrigue_lifestyle_xp_gain_mult = 0.2
    
    ruler_designer_cost = 20
    
    desc = {
        first_valid = {
            triggered_desc = {
                trigger = {
                    NOT = { exists = this }
                }
                desc = trait_education_intrigue_2_desc
            }
            desc = trait_education_intrigue_2_character_desc
        }
    }

    group = education_intrigue
    level = 2
}
"""

test_filepath = "test_events_parser_final.txt"
with open(test_filepath, "w", encoding="utf-8-sig") as f:
    f.write(test_content)
print(f"--- Created test file: {test_filepath} ---")
print("-" * 80)

parser = PdsParser()
print(f"--- Parsing '{test_filepath}'... ---")
parsed_nodes = parser.parse_file(test_filepath)

if parsed_nodes:
    print("\n--- Parsed Tree (Root Nodes Summary) ---")
    for i, node in enumerate(parsed_nodes):
        print(f"{i:03d}: {node!r}")
        if isinstance(node, PdsBlock) and node.children:
            # Limited child printing for brevity
            # pass 
            # To see more children:
            # print(f"     Children of '{node.key}' (first 5):")
            # for j, child_node in enumerate(node.children[:5]):
            #      print(f"       {j:03d}: {child_node!r}")
            # if len(node.children) > 5:
            #     print(f"       ... and {len(node.children) - 5} more children.")
            pass # Keep summary concise for now
    print("-" * 80)

    reconstructed_content = parser.to_string()
    
    # --- Detailed Diffing and Verification ---
    # Using splitlines(True) for difflib to handle newlines consistently if possible
    original_lines_for_diff = test_content.splitlines(True)
    reconstructed_lines_for_diff = reconstructed_content.splitlines(True)

    # Remove initial blank line from test_content if it exists and reconstruct doesn't make one
    if original_lines_for_diff and original_lines_for_diff[0].strip() == "":
        original_lines_for_diff = original_lines_for_diff[1:]
    if reconstructed_lines_for_diff and reconstructed_lines_for_diff[0].strip() == "":
         reconstructed_lines_for_diff = reconstructed_lines_for_diff[1:]

    diff = list(difflib.unified_diff(original_lines_for_diff, reconstructed_lines_for_diff,
                                     fromfile='Original', tofile='Reconstructed', lineterm='', n=3))
    
    # Check if the diff list contains any actual difference lines (starting with '+' or '-')
    # excluding the header lines '--- Original' and '+++ Reconstructed' and '@@ ... @@'
    actual_diff_lines = [d_line for d_line in diff if (d_line.startswith('+') or d_line.startswith('-')) and \
                                                    not d_line.startswith('---') and not d_line.startswith('+++')]

    if not actual_diff_lines:
        print("\nSUCCESS: Reconstructed content PERFECTLY matches original (or only whitespace/newline normalizations not caught by this diff).")
    else:
        print("\nWARNING: Reconstructed content differences found. Diff printed below:")
        print("Legend: '-' Original, '+' Reconstructed. Context lines are unchanged.")
        print("\n" + "="*70 + " DIFF OUTPUT " + "="*70)
        for line_diff in diff: # Print all diff lines including headers and context
            sys.stdout.write(line_diff) # Use sys.stdout.write to preserve exact line endings from diff
        print("="*70 + " END DIFF " + "="*70)

        recon_file = "reconstructed_parser_final_output.txt"
        orig_file = "original_parser_final_input.txt"
        with open(recon_file, "w", encoding="utf-8-sig") as f: f.write(reconstructed_content)
        with open(orig_file, "w", encoding="utf-8-sig") as f: f.write(test_content)
        print(f"\nFor detailed comparison, see '{orig_file}' and '{recon_file}'")
else:
    print(f"ERROR: No nodes parsed from {test_filepath}.")

# --- Test find_node (example) ---
if parsed_nodes:
    print("\n--- Testing find_node ---")
    test_block_for_find = next((n for n in parsed_nodes if isinstance(n, PdsBlock) and n.key == 'witch.1001'), None)
    if test_block_for_find:
        print(f"Searching in block: '{test_block_for_find.key}' (L{test_block_for_find.line_number})")
        path1 = 'trigger.is_witch_trigger'
        found_node1 = test_block_for_find.find_node(path1)
        print(f"Finding '{path1}': {found_node1!r}" + (f" | Value: '{found_node1.value}'" if hasattr(found_node1, 'value') else ""))
        path2 = 'trigger.any_relation.type'
        found_node2 = test_block_for_find.find_node(path2)
        print(f"Finding '{path2}': {found_node2!r}" + (f" | Value: '{found_node2.value}'" if hasattr(found_node2, 'value') else ""))
        path3 = 'immediate.if.limit.house.house_head' # Corrected path
        found_node3 = test_block_for_find.find_node(path3)
        print(f"Finding '{path3}': {found_node3!r}" + (f" | Value: '{found_node3.value}'" if hasattr(found_node3, 'value') else ""))

    target_key = 'scripted_trigger witch_1002_allow_reveal_outcome_trigger'
    scripted_trigger_node = next((n for n in parsed_nodes if hasattr(n, 'key') and n.key == target_key), None)
    if scripted_trigger_node:
        print(f"Found root node '{target_key}': {scripted_trigger_node!r}")
        if isinstance(scripted_trigger_node, PdsBlock):
            exists_node = scripted_trigger_node.find_node('exists')
            print(f"  Finding 'exists' in it: {exists_node!r}" + (f" | Value: '{exists_node.value}'" if hasattr(exists_node, 'value') else ""))
print("\n" + "="*80)
print("Parser Test Run Complete.")
print("="*80)

--- Initializing Test Environment ---
Current working directory: c:\Users\Galaxy\LEVI\jupyter\ck3_mod_update
--- Module 'pds_parser' reloaded. ---
--- Using PdsParser from: pds_parser.py ---
--------------------------------------------------------------------------------
--- Created test file: test_events_parser_final.txt ---
--------------------------------------------------------------------------------
--- Parsing 'test_events_parser_final.txt'... ---

--- Parsed Tree (Root Nodes Summary) ---
000: <PdsBlankLine L1 C1 Key='Blank Line'>
001: <PdsComment L2 C1 Key='Comment: '###############...''>
002: <PdsComment L3 C1 Key='Comment: 'WITCH EVENTS #...''>
003: <PdsComment L4 C1 Key='Comment: '###############...''>
004: <PdsBlankLine L5 C1 Key='Blank Line'>
005: <PdsKeyValuePair L6 C1 Key='namespace'>
006: <PdsBlankLine L7 C1 Key='Blank Line'>
007: <PdsComment L8 C1 Key='Comment: 'witch.1001-1999 - Guardian cov...''>
008: <PdsComment L9 C1 Key='Comment: 'witch.2001-2899 - Convert to w...

In [ ]:
## LEXER TEST  ##

import os
import sys
import importlib # Added for attempting to reload the module

# --- Configuration ---
LEXER_MODULE_NAME = 'pds_lexer' # Ensure your lexer file is pds_lexer.py

# --- Attempt to reload the lexer module ---
# This is to help ensure the latest version is used, but restarting the kernel is more reliable.
if LEXER_MODULE_NAME in sys.modules:
    try:
        lexer_module = importlib.reload(sys.modules[LEXER_MODULE_NAME])
        print(f"--- Module '{LEXER_MODULE_NAME}' reloaded successfully. ---")
    except Exception as e:
        print(f"--- FAILED to reload module '{LEXER_MODULE_NAME}': {e} ---")
else:
    print(f"--- Module '{LEXER_MODULE_NAME}' not yet imported, will import fresh. ---")

# --- Path setup ---
# Ensure pds_lexer.py is in the Python path
current_working_directory = os.getcwd()
if current_working_directory not in sys.path:
    sys.path.insert(0, current_working_directory)
print(f"--- Current working directory: {current_working_directory} ---")
print(f"--- Python sys.path (first few entries): {sys.path[:3]} ---")

# --- Import Lexer (after potential reload) ---
try:
    from pds_lexer import PdsLexer, PdsToken
    print("--- PdsLexer and PdsToken imported successfully. ---")
except ImportError as e:
    print(f"--- FATAL: Could not import PdsLexer or PdsToken: {e} ---")
    print("--- Please ensure 'pds_lexer.py' is in the same directory or Python path and contains these classes. ---")
    # Stop further execution if import fails
    raise

# --- Test Content ---
test_content = """
################
# WITCH EVENTS #
################

namespace = witch

#witch.1001-1999 - Guardian coverts ward
#witch.2001-2899 - Convert to witchcraft scheme

witch.1001 = { #by Mathilda Bjarnehed
    hidden = yes
    
    trigger = {
        is_witch_trigger = no
        any_relation = {
            type = guardian
            is_witch_trigger = yes
        }
    }

    immediate = {
        save_scope_as = child
        if = { # Conditional block
            limit = {
                is_ai = yes
                exists = house
                house = {
                    has_house_modifier = witch_coven
                    house_head = { is_ai = yes }
                }
                any_relation = {
                    type = guardian
                    is_ai = yes
                }
            }
            child_witch_conversion_success_effect = yes
        }
        else = {
            random_relation = { type = guardian trigger_event = witch.1002 }
        }
    }
}   

scripted_trigger witch_1002_allow_reveal_outcome_trigger = {
    exists = scope:child.liege
    scope:guardian = {
        NOT = { this = scope:child.liege }
        any_secret = {
            secret_type = secret_witch
            OR = {
                NOT = { is_known_by = scope:child }
                NOT = { is_known_by = scope:child.liege }
            }
        }
    }
}


# Standard Values
@pos_compat_high = 30
@pos_compat_medium = 15
@pos_compat_low = 5

# INTRIGUE OUTCOMES
education_intrigue_1 = {
    minimum_age = 16
    intrigue = 2
    category = education
    monthly_intrigue_lifestyle_xp_gain_mult = 0.1
    
    ruler_designer_cost = 0
    
    culture_modifier = {
        parameter = poorly_educated_leaders_distrusted
        feudal_government_opinion = -10
    }
    
    desc = {
        first_valid = {
            triggered_desc = {
                trigger = {
                    NOT = { exists = this }
                }
                desc = trait_education_intrigue_1_desc
            }
            desc = trait_education_intrigue_1_character_desc
        }
    }

    group = education_intrigue
    level = 1
}
education_intrigue_2 = {
    minimum_age = 16
    intrigue = 4
    category = education
    monthly_intrigue_lifestyle_xp_gain_mult = 0.2
    
    ruler_designer_cost = 20
    
    desc = {
        first_valid = {
            triggered_desc = {
                trigger = {
                    NOT = { exists = this }
                }
                desc = trait_education_intrigue_2_desc
            }
            desc = trait_education_intrigue_2_character_desc
        }
    }

    group = education_intrigue
    level = 2
}
"""

# --- Helper function to inspect characters ---
def inspect_text_at_location(text, target_line, target_col, window=20):
    """Inspects characters around a given line and column (1-indexed)."""
    print(f"\n--- Inspecting text_content around L{target_line}, C{target_col} ---")
    lines = text.splitlines(True) # Keep line endings
    if not (0 < target_line <= len(lines)):
        print(f"ERROR: Target line {target_line} is out of bounds (1-{len(lines)}).")
        return

    line_content = lines[target_line - 1]
    
    # Adjust column to be 0-indexed for string slicing
    char_index = target_col - 1

    if not (0 <= char_index < len(line_content)):
        print(f"ERROR: Target column {target_col} is out of bounds for line {target_line} (length {len(line_content)}).")
        print(f"Line content: '{line_content.rstrip()}'")
        return

    actual_char = line_content[char_index]
    print(f"Character at L{target_line}, C{target_col}: '{actual_char}' (ord: {ord(actual_char)}, hex: {hex(ord(actual_char))})")

    start = max(0, char_index - window // 2)
    end = min(len(line_content), char_index + window // 2 + 1)
    
    context_before = line_content[start:char_index]
    context_after = line_content[char_index+1:end]
    
    print(f"Context: '{context_before}<HERE>{actual_char}<HERE>{context_after.rstrip()}'")
    print(f"Line (raw): {repr(line_content)}")


# --- Lexer Test Execution ---
print("\n--- Starting Lexer Test ---")

# Inspect the suspected character location IN THE ORIGINAL test_content string
# This helps verify if the input string itself has an anomaly.
# The error is at line 11, col 16.
# Note: Line counting for `test_content` string literal starts after the initial triple quotes.
# The first actual content line "################" is line 2 if we count the initial blank line.
# Let's find the absolute character position for L11, C16 of the *parsed content*.
# The lexer's line counting starts at 1.
# The problematic line in the content block: "witch.1001 = { #by Mathilda Bjarnehed"

# To find the absolute position for `inspect_text_at_location`, we need to be careful.
# Let's assume the lexer's line 11 refers to the 11th non-empty or significant line.
# The content starts with a newline.
# 1: (empty)
# 2: ################
# 3: # WITCH EVENTS #
# 4: ################
# 5: (empty)
# 6: namespace = witch
# 7: (empty)
# 8: #witch.1001-1999 - Guardian coverts ward
# 9: #witch.2001-2899 - Convert to witchcraft scheme
#10: (empty)
#11: witch.1001 = { #by Mathilda Bjarnehed  <-- This is the line
inspect_text_at_location(test_content, 11, 16)


lexer = PdsLexer(test_content)
tokens = [] # Initialize tokens list

try:
    tokens = lexer.tokenize()
    print(f"--- Lexing Complete. Found {len(tokens)} tokens. ---")
    
    if tokens: # Check if tokens list is not empty
        print("\n--- First 20 Tokens: ---")
        for i, token in enumerate(tokens[:20]):
            print(f"{i:03d}: {token}")
        
        print("\n--- Last 20 Tokens (including EOF): ---")
        for i, token in enumerate(tokens[-20:], start=max(0, len(tokens)-20)):
            print(f"{i:03d}: {token}")

        # Example: Find tokens around a specific line where the error might be
        error_line = 11
        print(f"\n--- Tokens around Line {error_line} (and +/- 1 line): ---")
        for token in tokens:
            if error_line -1 <= token.line <= error_line + 1:
                print(token)
    else:
        print("--- No tokens were generated. ---")

except ValueError as e:
    print(f"LEXER ERROR: {e}")
    print("\n--- Lexer state at point of error (if accessible, depends on lexer structure): ---")
    print(f"Lexer Position: {getattr(lexer, 'pos', 'N/A')}")
    print(f"Lexer Line: {getattr(lexer, 'line', 'N/A')}")
    print(f"Lexer Column: {getattr(lexer, 'column', 'N/A')}")
    if hasattr(lexer, 'pos') and hasattr(lexer, 'text'):
        error_pos = lexer.pos
        text_context_start = max(0, error_pos - 30)
        text_context_end = min(len(lexer.text), error_pos + 30)
        print(f"Text context around error (pos {error_pos}):")
        print(f"...'{lexer.text[text_context_start:error_pos]}<ERROR_HERE>{lexer.text[error_pos:text_context_end]}'...")
except Exception as e_general:
    print(f"AN UNEXPECTED ERROR OCCURRED: {e_general}")
    import traceback
    traceback.print_exc()

print("\n--- Lexer Test Complete ---")

--- Module 'pds_lexer' not yet imported, will import fresh. ---
--- Current working directory: c:\Users\Galaxy\LEVI\jupyter\ck3_mod_update ---
--- Python sys.path (first few entries): ['c:\\Users\\Galaxy\\LEVI\\jupyter\\ck3_mod_update', 'c:\\Users\\Galaxy\\miniconda3\\python312.zip', 'c:\\Users\\Galaxy\\miniconda3\\DLLs'] ---
--- PdsLexer and PdsToken imported successfully. ---

--- Starting Lexer Test ---

--- Inspecting text_content around L11, C16 ---
Character at L11, C16: '#' (ord: 35, hex: 0x23)
Context: '.1001 = { <HERE>#<HERE>by Mathild'
Line (raw): 'witch.1001 = { #by Mathilda Bjarnehed\n'
--- DEBUGGING BLOCK IN PdsLexer ACTIVATED ---
LEXER_DEBUG: Current char: '#' (Unicode ord: 35), Line: 11, Column: 16, Pos: 177
LEXER_DEBUG: Text context: 'witch.1001 = { <HERE>#by Mathilda Bja'
LEXER_DEBUG: Slice for COMMENT match (first 30 chars): '#by Mathilda Bjarnehed
    hid'
LEXER_DEBUG: Direct test of COMMENT pattern SUCCEEDED. Group(0): '#by Mathilda Bjarnehed'
--- END LEXER DEBUGGIN

In [1]:
# Jupyter Notebook Cell

import os
import sys

# Ensure pds_parser.py and pds_differ.py are in the same directory
# Restart your Jupyter Notebook kernel before running this cell if files have changed!
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from pds_parser import PdsParser, PdsBlock, PdsKeyValuePair, PdsList, PdsComment, PdsBlankLine
from pds_differ import PdsDiffer, PdsChange

print(f"--- Successfully imported PdsParser from: {PdsParser.__module__}.py ---")
print(f"--- Successfully imported PdsDiffer from: {PdsDiffer.__module__}.py ---")
print("-" * 80)

# --- Define Test File Contents ---

# Test Case 1: Simple changes, additions, modifications
# Original state:
old_vanilla_content_1 = """
key_a = value_a
block_b = {
    param_b1 = 10
    param_b2 = text_b2
}
key_c = value_c_old
"""

# Mod's changes: added key_x, modified key_c, modified block_b (param_b1 changed)
mod_content_1 = """
key_a = value_a
block_b = {
    param_b1 = 20 # Mod changed this
    param_b2 = text_b2
    param_b3 = new_param_by_mod # Mod added this
}
key_c = value_c_mod # Mod changed this
key_x = value_x_by_mod # Mod added this
"""

# New Vanilla's changes: modified key_c, added key_y, modified block_b (param_b2 changed)
new_vanilla_content_1 = """
key_a = value_a
block_b = {
    param_b1 = 10
    param_b2 = text_b2_new # NV changed this
    param_b4 = new_param_by_nv # NV added this
}
key_c = value_c_nv # NV changed this (CONFLICT with mod)
key_y = value_y_by_nv # NV added this
"""

# Test Case 2: Deletions, Block type change (simplified)
# old_vanilla: has a block, mod deletes it, new vanilla changes it to a simple KV
old_vanilla_content_2 = """
file_version = 1.0
trait_block = {
    attr_a = 1
    attr_b = 2
}
event_id = 123
"""

# Mod's changes: trait_block is deleted, event_id modified
mod_content_2 = """
file_version = 1.0
event_id = 456_mod # Mod changed this
"""

# New Vanilla's changes: trait_block becomes a KV, event_id changed by NV
new_vanilla_content_2 = """
file_version = 1.1 # NV updated
trait_block = "simplified" # NV changed block to KV
event_id = 789_nv # NV changed this (CONFLICT with mod)
"""


# --- Helper Function to Run Diff ---
def run_and_print_diff(test_name, old_content, mod_content, new_content):
    print(f"\n{'='*20} RUNNING DIFF TEST: {test_name} {'='*20}")
    
    # Write to temporary files for parsing
    with open("temp_old.txt", "w", encoding="utf-8-sig") as f: f.write(old_content)
    with open("temp_mod.txt", "w", encoding="utf-8-sig") as f: f.write(mod_content)
    with open("temp_new.txt", "w", encoding="utf-8-sig") as f: f.write(new_content)

    # Parse files
    parser = PdsParser()
    old_nodes = parser.parse_file("temp_old.txt")
    mod_nodes = parser.parse_file("temp_mod.txt")
    new_nodes = parser.parse_file("temp_new.txt")

    if not old_nodes or not mod_nodes or not new_nodes:
        print(f"ERROR: Failed to parse one or more files for test '{test_name}'.")
        return

    # Run the differ
    differ = PdsDiffer()
    changes = differ.diff_nodes(old_nodes, mod_nodes, new_nodes)

    print(f"\n--- Detected Changes for '{test_name}' ({len(changes)} changes) ---")
    for change in changes:
        print(change)
    print(f"{'='*20} END DIFF TEST: {test_name} {'='*20}\n")

# --- Run Tests ---
run_and_print_diff("Test Case 1: Simple Changes", old_vanilla_content_1, mod_content_1, new_vanilla_content_1)
run_and_print_diff("Test Case 2: Deletions and Type Changes", old_vanilla_content_2, mod_content_2, new_vanilla_content_2)

# Clean up temp files (optional, but good practice)
try:
    os.remove("temp_old.txt")
    os.remove("temp_mod.txt")
    os.remove("temp_new.txt")
except OSError:
    pass # File might not exist if parsing failed

--- Successfully imported PdsParser from: pds_parser.py ---
--- Successfully imported PdsDiffer from: pds_differ.py ---
--------------------------------------------------------------------------------

==================== RUNNING DIFF TEST: Test Case 1: Simple Changes ====================

--- Detected Changes for 'Test Case 1: Simple Changes' (7 changes) ---
PdsChange(Type='MOD_MODIFIED                       ', Path='block_b.param_b1', Parent='block_b', 
          Nodes=[O:param_b1='10', M:param_b1='20', N:param_b1='10'])
PdsChange(Type='VANILLA_MODIFIED                   ', Path='block_b.param_b2', Parent='block_b', 
          Nodes=[O:param_b2='text_b2', M:param_b2='text_b2', N:param_b2='text_b2_new'])
PdsChange(Type='MOD_ADDED                          ', Path='block_b.param_b3', Parent='block_b', 
          Nodes=[O:ABSENT, M:param_b3='new_param_by_mod', N:ABSENT])
PdsChange(Type='VANILLA_ADDED                      ', Path='block_b.param_b4', Parent='block_b', 
          Nodes=[O:

In [1]:
import os
import sys
import difflib
from datetime import datetime

# Add the current directory to Python path to ensure local imports
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# IMPORTANT: Ensure your pds_parser.py and pds_differ.py are the LATEST versions.
# You MUST restart your Jupyter Notebook kernel before running this cell if you've changed those files!
from pds_parser import PdsParser, PdsBlock, PdsKeyValuePair, PdsList, PdsComment, PdsOperatorCondition 
from pds_differ import PdsDiffer, PdsChange

print(f"--- Successfully imported PdsParser from: {PdsParser.__module__}.py ---")
print(f"--- Successfully imported PdsDiffer from: {PdsDiffer.__module__}.py ---")
print("-" * 80)

# --- Define Paths to Your Real CK3 Files ---
# IMPORTANT: Replace these with the actual paths on your system
# Ensure these files exist and represent your mod, old vanilla, and new vanilla states.

SIEGE_EVENTS_MOD_PATH = r"C:\Users\Galaxy\Documents\Paradox Interactive\Crusader Kings III\mod\custom_changes\events\siege_events.txt"
SIEGE_EVENTS_OLD_VANILLA_PATH = r"C:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\old_ver\events\siege_events.txt"
SIEGE_EVENTS_NEW_VANILLA_PATH = r"C:\Program Files (x86)\Steam\steamapps\common\Crusader Kings III\game\events\siege_events.txt"

INNOVATIONS_MOD_PATH = r"C:\Users\Galaxy\Documents\Paradox Interactive\Crusader Kings III\mod\custom_changes\common\culture\innovations\00_tribal_innovations.txt"
INNOVATIONS_OLD_VANILLA_PATH = r"C:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\old_ver\common\culture\innovations\00_tribal_innovations.txt"
INNOVATIONS_NEW_VANILLA_PATH = r"C:\Program Files (x86)\Steam\steamapps\common\Crusader Kings III\game\common\culture\innovations\00_tribal_innovations.txt"


# Helper function to read file content
def get_file_content(filepath):
    try:
        with open(filepath, 'r', encoding='utf-8-sig') as f:
            return f.read()
    except UnicodeDecodeError:
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                return f.read()
        except Exception as e_inner:
            print(f"ERROR reading {filepath} with utf-8 fallback: {e_inner}", file=sys.stderr)
            return None
    except FileNotFoundError:
        print(f"WARNING: File not found at {filepath}", file=sys.stderr)
        return None
    except Exception as e:
        print(f"ERROR reading {filepath}: {e}", file=sys.stderr)
        return None

# Helper to write content to a file
def write_to_file(filepath, content):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    try:
        with open(filepath, 'w', encoding='utf-8-sig') as f:
            f.write(content)
        return True
    except Exception as e:
        print(f"ERROR writing to {filepath}: {e}", file=sys.stderr)
        return False

# --- Helper Functions for Tree Manipulation (Copied from mod_updater.py) ---
# These are needed for the simulated merge in the test script.
def find_node_by_path(root_nodes_list, key_path, differ_util): 
    if not key_path: return None
    current_nodes_to_search = root_nodes_list 

    for i, segment in enumerate(key_path):
        found_at_current_level = False
        for node in current_nodes_to_search: 
            node_identifier = differ_util._get_node_identifier(node)
            if node_identifier == segment:
                if i == len(key_path) - 1: 
                    return node 
                elif isinstance(node, PdsBlock): 
                    current_nodes_to_search = node.children 
                    found_at_current_level = True
                    break 
                else: return None 
        if not found_at_current_level: return None 
    return None 

def get_parent_node_by_path(root_nodes_list, key_path, differ_util): 
    if not key_path or len(key_path) < 1: return None, None
    if len(key_path) == 1: # Top-level node, its "parent" is the root_nodes_list itself
        return root_nodes_list, key_path[0] 

    parent_path = key_path[:-1] 
    child_identifier = key_path[-1]

    parent_block = find_node_by_path(root_nodes_list, parent_path, differ_util)
    
    if isinstance(parent_block, PdsBlock): 
        return parent_block, child_identifier
    return None, None 

# Helper Function to Run Diff (updated to output files)
def run_and_print_diff(test_name, old_filepath, mod_filepath, new_filepath):
    print(f"\n{'='*20} RUNNING DIFF TEST: {test_name} {'='*20}")
    
    # Define output directory for this test run
    test_output_dir = os.path.join(os.getcwd(), "test_output", test_name.replace(" ", "_").replace("(", "").replace(")", ""), datetime.now().strftime("%Y%m%d_%H%M%S"))
    os.makedirs(test_output_dir, exist_ok=True)
    print(f"Output files for this test will be saved to: {test_output_dir}")

    old_content_raw = get_file_content(old_filepath)
    mod_content_raw = get_file_content(mod_filepath)
    new_content_raw = get_file_content(new_filepath)

    # Save raw files to output for reference
    write_to_file(os.path.join(test_output_dir, os.path.basename(old_filepath).replace(".txt", "_OLD_RAW.txt")), old_content_raw or "")
    write_to_file(os.path.join(test_output_dir, os.path.basename(mod_filepath).replace(".txt", "_MOD_RAW.txt")), mod_content_raw or "")
    write_to_file(os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_NEW_RAW.txt")), new_content_raw or "")
    print(f"  Raw files saved: _OLD_RAW.txt, _MOD_RAW.txt, _NEW_RAW.txt")

    # Parse files
    parser = PdsParser()
    old_nodes = parser.parse_file(old_filepath)
    mod_nodes = parser.parse_file(mod_filepath)
    new_nodes = parser.parse_file(new_filepath)

    # Ensure empty lists if files were not found/parsed for diffing
    if old_nodes is None: old_nodes = []
    if mod_nodes is None: mod_nodes = []
    if new_nodes is None: new_nodes = []

    if not old_nodes and not mod_nodes and not new_nodes:
        print(f"SKIPPING: No content to parse for test '{test_name}' from any source (O, M, N). Check file paths.")
        return

    # Reconstruct parsed content to verify parser
    reconstructed_old = PdsParser._nodes_to_string(old_nodes)
    reconstructed_mod = PdsParser._nodes_to_string(mod_nodes)
    reconstructed_new = PdsParser._nodes_to_string(new_nodes)

    # Save reconstructed files
    write_to_file(os.path.join(test_output_dir, os.path.basename(old_filepath).replace(".txt", "_OLD_RECONSTRUCTED.txt")), reconstructed_old)
    write_to_file(os.path.join(test_output_dir, os.path.basename(mod_filepath).replace(".txt", "_MOD_RECONSTRUCTED.txt")), reconstructed_mod)
    write_to_file(os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_NEW_RECONSTRUCTED.txt")), reconstructed_new)
    print(f"  Reconstructed files saved: _OLD_RECONSTRUCTED.txt, _MOD_RECONSTRUCTED.txt, _NEW_RECONSTRUCTED.txt")

    # Check for perfect reconstruction (crucial for parser robustness)
    old_raw_strip = old_content_raw.strip() if old_content_raw else ""
    mod_raw_strip = mod_content_raw.strip() if mod_content_raw else ""
    new_raw_strip = new_content_raw.strip() if new_content_raw else ""

    if reconstructed_old.strip() != old_raw_strip:
        print(f"\nWARNING: Reconstruction mismatch for OLD file: {os.path.basename(old_filepath)}. Compare _OLD_RAW.txt and _OLD_RECONSTRUCTED.txt")
    if reconstructed_mod.strip() != mod_raw_strip:
        print(f"\nWARNING: Reconstruction mismatch for MOD file: {os.path.basename(mod_filepath)}. Compare _MOD_RAW.txt and _MOD_RECONSTRUCTED.txt")
    if reconstructed_new.strip() != new_raw_strip:
        print(f"\nWARNING: Reconstruction mismatch for NEW file: {os.path.basename(new_filepath)}. Compare _NEW_RAW.txt and _NEW_RECONSTRUCTED.txt")

    # Run the differ
    differ = PdsDiffer()
    changes = differ.diff_nodes(old_nodes, mod_nodes, new_nodes)

    print(f"\n--- DETECTED CHANGES for '{test_name}' ({len(changes)} changes) ---")
    if not changes:
        print("    No significant changes detected (or only identical comments/blank lines were filtered).")
    for change in changes:
        print(change) # PdsChange.__repr__ provides detailed formatting
    
    # --- SIMULATE MERGE (for display purposes only in this test script) ---
    simulated_merged_nodes = [node.copy() for node in new_nodes] # Deep copy new_nodes for modification
    
    # Pass simulated_merged_nodes as the target list
    def _simulate_apply_change(target_nodes_list_root, change_obj, differ_instance_for_helpers):
        parent_nodes_list_or_block, child_id_in_parent = get_parent_node_by_path(target_nodes_list_root, change_obj.key_path, differ_instance_for_helpers)
        
        if parent_nodes_list_or_block is None:
            print(f"  SIMULATED MERGE WARNING: Parent for '{'.'.join(change_obj.key_path)}' not found in target tree. Cannot apply change.")
            return

        sim_comment = f"SimulatedMerge:{datetime.now().strftime('%Y%m%d%H%M%S')}"

        if change_obj.type == 'MOD_ADDED':
            new_node = change_obj.mod_node.copy()
            if hasattr(new_node, 'comment_text_on_line') and new_node.comment_text_on_line is not None: new_node.comment_text_on_line = (new_node.comment_text_on_line + f" {sim_comment} MOD_ADDED")
            else: new_node.comment_text_on_line = f"{sim_comment} MOD_ADDED"
            
            if isinstance(parent_nodes_list_or_block, list): # Root level addition
                parent_nodes_list_or_block.append(new_node)
            else: # Nested addition
                parent_nodes_list_or_block.add_child_at_appropriate_location(new_node)
        elif change_obj.type == 'MOD_MODIFIED':
            new_node = change_obj.mod_node.copy()
            if hasattr(new_node, 'comment_text_on_line') and new_node.comment_text_on_line is not None: new_node.comment_text_on_line = (new_node.comment_text_on_line + f" {sim_comment} MOD_MODIFIED")
            else: new_node.comment_text_on_line = f"{sim_comment} MOD_MODIFIED"

            if isinstance(parent_nodes_list_or_block, list): # Root level modification
                for idx, node in enumerate(parent_nodes_list_or_block):
                    if differ_instance_for_helpers._get_node_identifier(node) == child_id_in_parent:
                        parent_nodes_list_or_block[idx] = new_node
                        break
            else: # Nested modification
                parent_nodes_list_or_block.replace_child(child_id_in_parent, new_node)
        elif change_obj.type == 'VANILLA_ADDED' or change_obj.type == 'VANILLA_MODIFIED':
            node_in_output = find_node_by_path(target_nodes_list_root, change_obj.key_path, differ_instance_for_helpers)
            if node_in_output:
                if hasattr(node_in_output, 'comment_text_on_line') and node_in_output.comment_text_on_line is not None:
                    node_in_output.comment_text_on_line = (node_in_output.comment_text_on_line + f" {sim_comment} NV_MODIFIED")
                else: node_in_output.comment_text_on_line = f"{sim_comment} NV_MODIFIED"
        elif change_obj.type == 'CONFLICT_MODIFIED' or change_obj.type == 'CONFLICT_ADDITION':
            chosen_node = change_obj.mod_node.copy() # Auto-choose MOD's version for simulation
            if hasattr(chosen_node, 'comment_text_on_line') and chosen_node.comment_text_on_line is not None: chosen_node.comment_text_on_line = (chosen_node.comment_text_on_line + f" {sim_comment} CONFLICT_MOD_CHOSEN")
            else: chosen_node.comment_text_on_line = f"{sim_comment} CONFLICT_MOD_CHOSEN"

            if isinstance(parent_nodes_list_or_block, list):
                parent_nodes_list_or_block[:] = [n for n in parent_nodes_list_or_block if differ_instance_for_helpers._get_node_identifier(n) != child_id_in_parent]
                parent_nodes_list_or_block.append(chosen_node)
            else:
                if not parent_nodes_list_or_block.replace_child(child_id_in_parent, chosen_node):
                    parent_nodes_list_or_block.add_child_at_appropriate_location(chosen_node)
        elif change_obj.type in ['MOD_DELETED', 'VANILLA_DELETED', 'MOD_DELETED_VANILLA_ALSO_DELETED', 'CONFLICT_DELETION']:
            # For simulation, auto-delete
            if isinstance(parent_nodes_list_or_block, list):
                parent_nodes_list_or_block[:] = [n for n in parent_nodes_list_or_block if differ_instance_for_helpers._get_node_identifier(n) != child_id_in_parent]
            else:
                parent_nodes_list_or_block.remove_child(child_id_in_parent)

    for change in changes:
        _simulate_apply_change(simulated_merged_nodes, change, differ)

    simulated_merged_content = PdsParser._nodes_to_string(simulated_merged_nodes)
    write_to_file(os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_SIMULATED_MERGED.txt")), simulated_merged_content)
    print(f"  Simulated merged content saved to: {os.path.basename(new_filepath).replace('.txt', '_SIMULATED_MERGED.txt')}")
    
    print(f"\n--- DIFF: SIMULATED MERGED vs NEW VANILLA RAW for '{test_name}' (Output to _MERGED_VS_NEW_RAW.diff) ---")
    diff_filename = os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_MERGED_VS_NEW_RAW.diff"))
    with open(diff_filename, 'w', encoding='utf-8') as f_diff:
        diff_lines = list(difflib.unified_diff(
            (new_content_raw if new_content_raw else "").strip().splitlines(keepends=True),
            simulated_merged_content.strip().splitlines(keepends=True),
            fromfile='NEW_VANILLA_RAW', tofile='SIMULATED_MERGED_OUTPUT', lineterm=''
        ))
        if diff_lines:
            f_diff.writelines(diff_lines)
            print(f"  Diff saved to: {os.path.basename(diff_filename)}")
        else:
            print("  SIMULATED MERGED content is identical to RAW NEW VANILLA (no meaningful changes applied by merge logic in this simulation).")
    print("-" * 80)


# --- Run Tests with Your Real Files ---
print("Running diff tests with real CK3 files.")

# Test 1: siege_events.txt
run_and_print_diff("Siege Events (real files)", 
                   SIEGE_EVENTS_OLD_VANILLA_PATH, 
                   SIEGE_EVENTS_MOD_PATH, 
                   SIEGE_EVENTS_NEW_VANILLA_PATH)

# Test 2: 00_tribal_innovations.txt
run_and_print_diff("00_tribal_innovations (real files)",
                   INNOVATIONS_OLD_VANILLA_PATH,
                   INNOVATIONS_MOD_PATH,
                   INNOVATIONS_NEW_VANILLA_PATH)

print("\n--- Real File Diff Tests Complete ---")

--- Successfully imported PdsParser from: pds_parser.py ---
--- Successfully imported PdsDiffer from: pds_differ.py ---
--------------------------------------------------------------------------------
Running diff tests with real CK3 files.

==================== RUNNING DIFF TEST: Siege Events (real files) ====================
Output files for this test will be saved to: c:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\test_output\Siege_Events_real_files\20250526_163245
  Raw files saved: _OLD_RAW.txt, _MOD_RAW.txt, _NEW_RAW.txt
  Reconstructed files saved: _OLD_RECONSTRUCTED.txt, _MOD_RECONSTRUCTED.txt, _NEW_RECONSTRUCTED.txt




--- DETECTED CHANGES for 'Siege Events (real files)' (12 changes) ---
PdsChange(Type='VANILLA_DELETED                    ', Path='siege.0002 =.after =', Parent='siege.0002 =', 
          Nodes=[O:after =={...}, M:after =={...}, N:ABSENT])
PdsChange(Type='VANILLA_ADDED                      ', Path='siege.0002 =.immediate =.remove_variable', Parent='siege.0002 =.imm

In [ ]:
import os
import sys
import difflib
from datetime import datetime

# Add the current directory to Python path to ensure local imports
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# IMPORTANT: Ensure your pds_parser.py and pds_differ.py are the LATEST versions.
# You MUST restart your Jupyter Notebook kernel before running this cell if you've changed those files!
from pds_parser import PdsParser, PdsBlock, PdsKeyValuePair, PdsList, PdsComment, PdsOperatorCondition 
from pds_differ import PdsDiffer, PdsChange
from pds_postprocessor import PdsPostProcessor # Import the new post-processor

print(f"--- Successfully imported PdsParser from: {PdsParser.__module__}.py ---")
print(f"--- Successfully imported PdsDiffer from: {PdsDiffer.__module__}.py ---")
print(f"--- Successfully imported PdsPostProcessor from: {PdsPostProcessor.__module__}.py ---") # New print
print("-" * 80)

# --- Define Paths to Your Real CK3 Files ---
# IMPORTANT: Replace these with the actual paths on your system
# Ensure these files exist and represent your mod, old vanilla, and new vanilla states.

SIEGE_EVENTS_MOD_PATH = r"C:\Users\Galaxy\Documents\Paradox Interactive\Crusader Kings III\mod\custom_changes\events\siege_events.txt"
SIEGE_EVENTS_OLD_VANILLA_PATH = r"C:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\old_ver\events\siege_events.txt"
SIEGE_EVENTS_NEW_VANILLA_PATH = r"C:\Program Files (x86)\Steam\steamapps\common\Crusader Kings III\game\events\siege_events.txt"

INNOVATIONS_MOD_PATH = r"C:\Users\Galaxy\Documents\Paradox Interactive\Crusader Kings III\mod\custom_changes\common\culture\innovations\00_tribal_innovations.txt"
INNOVATIONS_OLD_VANILLA_PATH = r"C:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\old_ver\common\culture\innovations\00_tribal_innovations.txt"
INNOVATIONS_NEW_VANILLA_PATH = r"C:\Program Files (x86)\Steam\steamapps\common\Crusader Kings III\game\common\culture\innovations\00_tribal_innovations.txt"


# Helper function to read file content
def get_file_content(filepath):
    try:
        with open(filepath, 'r', encoding='utf-8-sig') as f:
            return f.read()
    except UnicodeDecodeError:
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                return f.read()
        except Exception as e_inner:
            print(f"ERROR reading {filepath} with utf-8 fallback: {e_inner}", file=sys.stderr)
            return None
    except FileNotFoundError:
        print(f"WARNING: File not found at {filepath}", file=sys.stderr)
        return None
    except Exception as e:
        print(f"ERROR reading {filepath}: {e}", file=sys.stderr)
        return None

# Helper to write content to a file
def write_to_file(filepath, content):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    try:
        with open(filepath, 'w', encoding='utf-8-sig') as f:
            f.write(content)
        return True
    except Exception as e:
        print(f"ERROR writing to {filepath}: {e}", file=sys.stderr)
        return False

# --- Helper Functions for Tree Manipulation (Copied from mod_updater.py) ---
# These are needed for the simulated merge in the test script.
def find_node_by_path(root_nodes_list, key_path, differ_util): 
    if not key_path: return None
    current_nodes_to_search = root_nodes_list 

    for i, segment in enumerate(key_path):
        found_at_current_level = False
        for node in current_nodes_to_search: 
            node_identifier = differ_util._get_node_identifier(node)
            if node_identifier == segment:
                if i == len(key_path) - 1: 
                    return node 
                elif isinstance(node, PdsBlock): 
                    current_nodes_to_search = node.children 
                    found_at_current_level = True
                    break 
                else: return None 
        if not found_at_current_level: return None 
    return None 

def get_parent_node_by_path(root_nodes_list, key_path, differ_util): 
    if not key_path or len(key_path) < 1: return None, None
    if len(key_path) == 1: # Top-level node, its "parent" is the root_nodes_list itself
        return root_nodes_list, key_path[0] 

    parent_path = key_path[:-1] 
    child_identifier = key_path[-1]

    parent_block = find_node_by_path(root_nodes_list, parent_path, differ_util)
    
    if isinstance(parent_block, PdsBlock): 
        return parent_block, child_identifier
    return None, None 

# Helper to normalize content for comparison purposes (e.g., when comparing raw vs reconstructed)
# This accounts for the post-processor's behavior of reducing multiple blank lines to a single blank line.
def normalize_for_comparison(text):
    if not text: return ""
    # Reduce any sequence of 3 or more newlines to exactly two newlines (a single blank line)
    # The post-processor reduces \n(\s*\n){2,} to \n\n
    # So, \n\n\n -> \n\n, \n\n\n\n -> \n\n, etc.
    # This regex handles one or more newlines followed by two or more potential newlines with whitespace.
    normalized = re.sub(r'\n(\s*\n)+(\s*\n)+', '\n\n', text)
    # Also strip any leading/trailing whitespace including newlines, as the parser output usually doesn't have them
    return normalized.strip()

# Helper Function to Run Diff (updated to output files)
def run_and_print_diff(test_name, old_filepath, mod_filepath, new_filepath):
    print(f"\n{'='*20} RUNNING DIFF TEST: {test_name} {'='*20}")
    
    # Define output directory for this test run
    test_output_dir = os.path.join(os.getcwd(), "test_output", test_name.replace(" ", "_").replace("(", "").replace(")", ""), datetime.now().strftime("%Y%m%d_%H%M%S"))
    os.makedirs(test_output_dir, exist_ok=True)
    print(f"Output files for this test will be saved to: {test_output_dir}")

    old_content_raw = get_file_content(old_filepath)
    mod_content_raw = get_file_content(mod_filepath)
    new_content_raw = get_file_content(new_filepath)

    # Save raw files to output for reference
    write_to_file(os.path.join(test_output_dir, os.path.basename(old_filepath).replace(".txt", "_OLD_RAW.txt")), old_content_raw or "")
    write_to_file(os.path.join(test_output_dir, os.path.basename(mod_filepath).replace(".txt", "_MOD_RAW.txt")), mod_content_raw or "")
    write_to_file(os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_NEW_RAW.txt")), new_content_raw or "")
    print(f"  Raw files saved: _OLD_RAW.txt, _MOD_RAW.txt, _NEW_RAW.txt")

    # Parse files
    parser = PdsParser()
    old_nodes = parser.parse_file(old_filepath)
    mod_nodes = parser.parse_file(mod_filepath)
    new_nodes = parser.parse_file(new_filepath)

    # Ensure empty lists if files were not found/parsed for diffing
    if old_nodes is None: old_nodes = []
    if mod_nodes is None: mod_nodes = []
    if new_nodes is None: new_nodes = []

    if not old_nodes and not mod_nodes and not new_nodes:
        print(f"SKIPPING: No content to parse for test '{test_name}' from any source (O, M, N). Check file paths.")
        return

    # Reconstruct parsed content to verify parser (DO NOT apply post-processor here)
    reconstructed_old = PdsParser._nodes_to_string(old_nodes)
    reconstructed_mod = PdsParser._nodes_to_string(mod_nodes)
    reconstructed_new = PdsParser._nodes_to_string(new_nodes)

    # Save reconstructed files
    write_to_file(os.path.join(test_output_dir, os.path.basename(old_filepath).replace(".txt", "_OLD_RECONSTRUCTED.txt")), reconstructed_old)
    write_to_file(os.path.join(test_output_dir, os.path.basename(mod_filepath).replace(".txt", "_MOD_RECONSTRUCTED.txt")), reconstructed_mod)
    write_to_file(os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_NEW_RECONSTRUCTED.txt")), reconstructed_new)
    print(f"  Reconstructed files saved: _OLD_RECONSTRUCTED.txt, _MOD_RECONSTRUCTED.txt, _NEW_RECONSTRUCTED.txt")

    # Check for reconstruction consistency (after normalizing raw content for comparison)
    normalized_old_raw = normalize_for_comparison(old_content_raw)
    normalized_mod_raw = normalize_for_comparison(mod_content_raw)
    normalized_new_raw = normalize_for_comparison(new_content_raw)

    if reconstructed_old.strip() != normalized_old_raw:
        print(f"\nWARNING: Reconstruction mismatch for OLD file: {os.path.basename(old_filepath)}. Parser output might differ slightly in whitespace. Compare _OLD_RAW.txt and _OLD_RECONSTRUCTED.txt using a diff tool (or check _OLD_RECONSTRUCTED_VS_NORMALIZED_RAW.diff).")
        with open(os.path.join(test_output_dir, os.path.basename(old_filepath).replace(".txt", "_OLD_RECONSTRUCTED_VS_NORMALIZED_RAW.diff")), 'w', encoding='utf-8') as f_diff:
            diff_lines = list(difflib.unified_diff(
                normalized_old_raw.splitlines(keepends=True),
                reconstructed_old.strip().splitlines(keepends=True),
                fromfile='NORMALIZED_RAW', tofile='RECONSTRUCTED', lineterm=''
            ))
            f_diff.writelines(diff_lines)

    if reconstructed_mod.strip() != normalized_mod_raw:
        print(f"\nWARNING: Reconstruction mismatch for MOD file: {os.path.basename(mod_filepath)}. Parser output might differ slightly in whitespace. Compare _MOD_RAW.txt and _MOD_RECONSTRUCTED.txt using a diff tool (or check _MOD_RECONSTRUCTED_VS_NORMALIZED_RAW.diff).")
        with open(os.path.join(test_output_dir, os.path.basename(mod_filepath).replace(".txt", "_MOD_RECONSTRUCTED_VS_NORMALIZED_RAW.diff")), 'w', encoding='utf-8') as f_diff:
            diff_lines = list(difflib.unified_diff(
                normalized_mod_raw.splitlines(keepends=True),
                reconstructed_mod.strip().splitlines(keepends=True),
                fromfile='NORMALIZED_RAW', tofile='RECONSTRUCTED', lineterm=''
            ))
            f_diff.writelines(diff_lines)

    if reconstructed_new.strip() != normalized_new_raw:
        print(f"\nWARNING: Reconstruction mismatch for NEW file: {os.path.basename(new_filepath)}. Parser output might differ slightly in whitespace. Compare _NEW_RAW.txt and _NEW_RECONSTRUCTED.txt using a diff tool (or check _NEW_RECONSTRUCTED_VS_NORMALIZED_RAW.diff).")
        with open(os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_NEW_RECONSTRUCTED_VS_NORMALIZED_RAW.diff")), 'w', encoding='utf-8') as f_diff:
            diff_lines = list(difflib.unified_diff(
                normalized_new_raw.splitlines(keepends=True),
                reconstructed_new.strip().splitlines(keepends=True),
                fromfile='NORMALIZED_RAW', tofile='RECONSTRUCTED', lineterm=''
            ))
            f_diff.writelines(diff_lines)

    # Run the differ
    differ = PdsDiffer()
    changes = differ.diff_nodes(old_nodes, mod_nodes, new_nodes) # Diff is still on AST nodes, not strings

    print(f"\n--- DETECTED CHANGES for '{test_name}' ({len(changes)} changes) ---")
    if not changes:
        print("    No significant changes detected (or only identical comments/blank lines were filtered).")
    for change in changes:
        print(change) # PdsChange.__repr__ provides detailed formatting
    
    # --- SIMULATE MERGE (for display purposes only in this test script) ---
    simulated_merged_nodes = [node.copy() for node in new_nodes] # Deep copy new_nodes for modification
    
    # Pass simulated_merged_nodes as the target list
    def _simulate_apply_change(target_nodes_list_root, change_obj, differ_instance_for_helpers):
        parent_nodes_list_or_block, child_id_in_parent = get_parent_node_by_path(target_nodes_list_root, change_obj.key_path, differ_instance_for_helpers)
        
        if parent_nodes_list_or_block is None:
            print(f"  SIMULATED MERGE WARNING: Parent for '{'.'.join(change_obj.key_path)}' not found in target tree. Cannot apply change.")
            return

        sim_comment = f"SimulatedMerge:{datetime.now().strftime('%Y%m%d%H%M%S')}"

        if change_obj.type == 'MOD_ADDED':
            new_node = change_obj.mod_node.copy()
            if hasattr(new_node, 'comment_text_on_line') and new_node.comment_text_on_line is not None: new_node.comment_text_on_line = (new_node.comment_text_on_line + f" {sim_comment} MOD_ADDED")
            else: new_node.comment_text_on_line = f"{sim_comment} MOD_ADDED"
            
            if isinstance(parent_nodes_list_or_block, list): # Root level addition
                parent_nodes_list_or_block.append(new_node)
            else: # Nested addition
                parent_nodes_list_or_block.add_child_at_appropriate_location(new_node)
        elif change_obj.type == 'MOD_MODIFIED':
            new_node = change_obj.mod_node.copy()
            if hasattr(new_node, 'comment_text_on_line') and new_node.comment_text_on_line is not None: new_node.comment_text_on_line = (new_node.comment_text_on_line + f" {sim_comment} MOD_MODIFIED")
            else: new_node.comment_text_on_line = f"{sim_comment} MOD_MODIFIED"

            if isinstance(parent_nodes_list_or_block, list): # Root level modification
                for idx, node in enumerate(parent_nodes_list_or_block):
                    if differ_instance_for_helpers._get_node_identifier(node) == child_id_in_parent:
                        parent_nodes_list_or_block[idx] = new_node
                        break
            else: # Nested modification
                parent_nodes_list_or_block.replace_child(child_id_in_parent, new_node)
        elif change_obj.type == 'VANILLA_ADDED' or change_obj.type == 'VANILLA_MODIFIED':
            node_in_output = find_node_by_path(target_nodes_list_root, change_obj.key_path, differ_instance_for_helpers)
            if node_in_output:
                if hasattr(node_in_output, 'comment_text_on_line') and node_in_output.comment_text_on_line is not None:
                    node_in_output.comment_text_on_line = (node_in_output.comment_text_on_line + f" {sim_comment} NV_MODIFIED")
                else: node_in_output.comment_text_on_line = f"{sim_comment} NV_MODIFIED"
        elif change_obj.type == 'CONFLICT_MODIFIED' or change_obj.type == 'CONFLICT_ADDITION':
            chosen_node = change_obj.mod_node.copy() # Auto-choose MOD's version for simulation
            if hasattr(chosen_node, 'comment_text_on_line') and chosen_node.comment_text_on_line is not None: chosen_node.comment_text_on_line = (chosen_node.comment_text_on_line + f" {sim_comment} CONFLICT_MOD_CHOSEN")
            else: chosen_node.comment_text_on_line = f"{sim_comment} CONFLICT_MOD_CHOSEN"

            if isinstance(parent_nodes_list_or_block, list):
                parent_nodes_list_or_block[:] = [n for n in parent_nodes_list_or_block if differ_instance_for_helpers._get_node_identifier(n) != child_id_in_parent]
                parent_nodes_list_or_block.append(chosen_node)
            else:
                if not parent_nodes_list_or_block.replace_child(child_id_in_parent, chosen_node):
                    parent_nodes_list_or_block.add_child_at_appropriate_location(chosen_node)
        elif change_obj.type in ['MOD_DELETED', 'VANILLA_DELETED', 'MOD_DELETED_VANILLA_ALSO_DELETED', 'CONFLICT_DELETION']:
            # For simulation, auto-delete
            if isinstance(parent_nodes_list_or_block, list):
                parent_nodes_list_or_block[:] = [n for n in parent_nodes_list_or_block if differ_instance_for_helpers._get_node_identifier(n) != child_id_in_parent]
            else:
                parent_nodes_list_or_block.remove_child(child_id_in_parent)

    for change in changes:
        _simulate_apply_change(simulated_merged_nodes, change, differ)

    simulated_merged_content = PdsParser._nodes_to_string(simulated_merged_nodes)
    # *** Apply Post-Processing to merged output ***
    post_processor = PdsPostProcessor() 
    simulated_merged_content = post_processor.process(simulated_merged_content) 

    write_to_file(os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_SIMULATED_MERGED.txt")), simulated_merged_content)
    print(f"  Simulated merged content saved to: {os.path.basename(new_filepath).replace('.txt', '_SIMULATED_MERGED.txt')}")
    
    print(f"\n--- DIFF: SIMULATED MERGED vs NEW VANILLA RAW for '{test_name}' (Output to _MERGED_VS_NEW_RAW.diff) ---")
    diff_filename = os.path.join(test_output_dir, os.path.basename(new_filepath).replace(".txt", "_MERGED_VS_NEW_RAW.diff"))
    with open(diff_filename, 'w', encoding='utf-8') as f_diff:
        # Compare post-processed merged output against a normalized version of the raw new vanilla content
        diff_lines = list(difflib.unified_diff(
            normalize_for_comparison(new_content_raw).splitlines(keepends=True),
            simulated_merged_content.strip().splitlines(keepends=True),
            fromfile='NORMALIZED_NEW_VANILLA_RAW', tofile='SIMULATED_MERGED_OUTPUT', lineterm=''
        ))
        if diff_lines:
            f_diff.writelines(diff_lines)
            print(f"  Diff saved to: {os.path.basename(diff_filename)}")
        else:
            print("  SIMULATED MERGED content is identical to NORMALIZED NEW VANILLA (merge logic applied and formatting normalized).")
    print("-" * 80)


def test_post_processor_fidelity(test_name, input_filepath, expected_output_filepath=None):
    """
    Tests the PdsPostProcessor in isolation by parsing a file, reconstructing it,
    and then applying the post-processor, comparing the result to an expected output.
    If no explicit expected_output_filepath is provided, it compares against a normalized
    version of the raw input.
    """
    print(f"\n{'='*20} RUNNING POST-PROCESSOR FIDELITY TEST: {test_name} {'='*20}")
    
    test_output_dir = os.path.join(os.getcwd(), "test_output", f"PostProcessorTest_{test_name.replace(' ', '_')}", datetime.now().strftime("%Y%m%d_%H%M%S"))
    os.makedirs(test_output_dir, exist_ok=True)
    print(f"Output files for this test will be saved to: {test_output_dir}")

    raw_input_content = get_file_content(input_filepath)
    if raw_input_content is None:
        print(f"SKIPPING: Input file not found for post-processor test: {input_filepath}")
        return

    write_to_file(os.path.join(test_output_dir, os.path.basename(input_filepath).replace(".txt", "_RAW_INPUT.txt")), raw_input_content)
    print(f"  Raw input content saved to: {os.path.basename(input_filepath).replace('.txt', '_RAW_INPUT.txt')}")

    # 1. Parse the raw input (this will produce the parser's standard output with its formatting)
    parser = PdsParser()
    parsed_nodes = parser.parse_file(input_filepath)
    parser_reconstructed_content = PdsParser._nodes_to_string(parsed_nodes)
    write_to_file(os.path.join(test_output_dir, os.path.basename(input_filepath).replace(".txt", "_PARSED_RECONSTRUCTED.txt")), parser_reconstructed_content)
    print(f"  Parsed and reconstructed content saved to: {os.path.basename(input_filepath).replace('.txt', '_PARSED_RECONSTRUCTED.txt')}")

    # 2. Apply the post-processor to the parser's output
    post_processor = PdsPostProcessor()
    post_processed_content = post_processor.process(parser_reconstructed_content)
    write_to_file(os.path.join(test_output_dir, os.path.basename(input_filepath).replace(".txt", "_POST_PROCESSED.txt")), post_processed_content)
    print(f"  Post-processed content saved to: {os.path.basename(input_filepath).replace('.txt', '_POST_PROCESSED.txt')}")

    # 3. Determine comparison target
    comparison_target_content = None
    target_label = ""
    if expected_output_filepath and os.path.exists(expected_output_filepath):
        comparison_target_content = get_file_content(expected_output_filepath)
        target_label = "EXPLICIT_EXPECTED_OUTPUT"
        write_to_file(os.path.join(test_output_dir, os.path.basename(expected_output_filepath).replace(".txt", "_EXPECTED_OUTPUT.txt")), comparison_target_content)
        print(f"  Comparing against explicit expected output: {os.path.basename(expected_output_filepath)}")
    else:
        # If no explicit expected output, compare against a normalized version of the original raw input
        comparison_target_content = normalize_for_comparison(raw_input_content)
        target_label = "NORMALIZED_RAW_INPUT"
        write_to_file(os.path.join(test_output_dir, os.path.basename(input_filepath).replace(".txt", "_NORMALIZED_RAW_INPUT.txt")), comparison_target_content)
        print(f"  No explicit expected output, comparing against normalized raw input.")


    if comparison_target_content is None:
        print(f"  SKIPPING COMPARISON: No valid content for comparison target.")
        return

    diff_filename = os.path.join(test_output_dir, "post_processed_vs_target.diff")
    with open(diff_filename, 'w', encoding='utf-8') as f_diff:
        diff_lines = list(difflib.unified_diff(
            (comparison_target_content if comparison_target_content else "").strip().splitlines(keepends=True),
            post_processed_content.strip().splitlines(keepends=True),
            fromfile=target_label, tofile='POST_PROCESSED_OUTPUT', lineterm=''
        ))
        if diff_lines:
            f_diff.writelines(diff_lines)
            print(f"  Diff saved to: {os.path.basename(diff_filename)}")
        else:
            print("  POST-PROCESSED content is identical to the comparison target (perfect formatting).")
    print("-" * 80)


# --- Run Tests with Your Real Files ---
print("Running diff tests with real CK3 files.")

# Test 1: siege_events.txt
run_and_print_diff("Siege Events (real files)", 
                   SIEGE_EVENTS_OLD_VANILLA_PATH, 
                   SIEGE_EVENTS_MOD_PATH, 
                   SIEGE_EVENTS_NEW_VANILLA_PATH)

# Test 2: 00_tribal_innovations.txt
run_and_print_diff("00_tribal_innovations (real files)",
                   INNOVATIONS_OLD_VANILLA_PATH,
                   INNOVATIONS_MOD_PATH,
                   INNOVATIONS_NEW_VANILLA_PATH)

# --- Dedicated Post-Processor Fidelity Tests ---
# Use one of your actual files as input. If you have an ideal formatted version,
# you can provide it as expected_output_filepath. Otherwise, it compares against
# a version of the raw input with its blank lines normalized.
print("\nRunning dedicated Post-Processor Fidelity Tests.")
test_post_processor_fidelity("Siege Events Post-Processing Test",
                             SIEGE_EVENTS_NEW_VANILLA_PATH,
                             # Optional: r"C:\path\to\your\ideal_siege_events_formatted.txt"
                            )

test_post_processor_fidelity("Innovations Post-Processing Test",
                             INNOVATIONS_NEW_VANILLA_PATH,
                             # Optional: r"C:\path\to\your\ideal_innovations_formatted.txt"
                            )

print("\n--- All Tests Complete ---")

--- Successfully imported PdsParser from: pds_parser.py ---
--- Successfully imported PdsDiffer from: pds_differ.py ---
--- Successfully imported PdsPostProcessor from: pds_postprocessor.py ---
--------------------------------------------------------------------------------
Running diff tests with real CK3 files.

==================== RUNNING DIFF TEST: Siege Events (real files) ====================
Output files for this test will be saved to: c:\Users\Galaxy\LEVI\jupyter\ck3_mod_update\test_output\Siege_Events_real_files\20250526_163122
  Raw files saved: _OLD_RAW.txt, _MOD_RAW.txt, _NEW_RAW.txt
  Reconstructed files saved: _OLD_RECONSTRUCTED.txt, _MOD_RECONSTRUCTED.txt, _NEW_RECONSTRUCTED.txt




--- DETECTED CHANGES for 'Siege Events (real files)' (12 changes) ---
PdsChange(Type='VANILLA_DELETED                    ', Path='siege.0002 =.after =', Parent='siege.0002 =', 
          Nodes=[O:after =={...}, M:after =={...}, N:ABSENT])
PdsChange(Type='VANILLA_ADDED                      ',